# Prepare the ten-year price panel

Run this notebook before [02_sector_dependence.ipynb](02_sector_dependence.ipynb). It filters the TITLON export, attaches the saved ICB classifications, and builds an observed adjusted-price panel with one share class per issuer. All three CSVs are written to `Data/eligible/`.


## Filter dates

Exclude XOSL non-trading dates over the last 15 years. Use the dates that remain in the last ten years to measure coverage. [XOSL calendar](https://github.com/gerrymanoim/exchange_calendars/blob/4.13.2/exchange_calendars/exchange_calendar_xosl.py) · [Euronext holiday calendars](https://www.euronext.com/en/trading/trading-hours-holidays) · [2020 year-end notice](https://connect2.euronext.com/sites/default/files/2020-11/IF201124CADE%202020%20Updated%20Holiday%20Calendar.pdf)

In [ ]:
import csv
from pathlib import Path
from tempfile import NamedTemporaryFile

import exchange_calendars as xcals
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
data_dir = root / "Data"
eligible_dir = data_dir / "eligible"
raw_source = data_dir / "BMG9156K1018_10.09.2026_14.59.34.csv"
eligible_path = eligible_dir / f"{raw_source.stem}_trading_days_eligible.csv"
identifiers = ["CompanyId", "Internal code"]
columns = ["Date", "ISIN", "AdjustedPrice", *identifiers]
rows = pd.read_csv(raw_source, sep=";", usecols=columns, dtype="string")
rows["Date"] = pd.to_datetime(rows["Date"], format="%Y-%m-%d")

end = rows["Date"].max()
calendar_start = end - pd.DateOffset(years=15)
window_start = end - pd.DateOffset(years=10)
calendar = xcals.get_calendar("XOSL", start=calendar_start, end=end)
non_trading_days = pd.date_range(calendar_start, end).difference(calendar.sessions)
on_closed_day = rows["Date"].isin(non_trading_days)

history = rows.loc[~on_closed_day]
window = history.loc[history["Date"].between(window_start, end)]
trading_dates = pd.DatetimeIndex(window["Date"].unique()).sort_values()

print(f"Excluded {on_closed_day.sum():,} rows on {rows.loc[on_closed_day, 'Date'].nunique()} closure dates.")
print(f"Coverage window: {window_start.date()} to {end.date()} ({len(trading_dates):,} observed dates).")

## Link ISINs and check coverage

Link ISINs transitively by nonmissing `CompanyId` or `Internal code` across the filtered history. Count a date once per company when any linked ISIN has a positive finite `AdjustedPrice`. Require a longest gap of at most five trading dates, including at the window boundaries. The stricter 5% missingness and final-session screens are applied in `02_sector_dependence.ipynb`.

In [ ]:
parent = {isin: isin for isin in history["ISIN"].unique()}


def company_root(isin: str) -> str:
    """Return the representative ISIN of a linked group."""
    while isin != parent[isin]:
        parent[isin] = parent[parent[isin]]
        isin = parent[isin]
    return isin


for column in identifiers:
    first_isin_by_id = {}
    links = history[["ISIN", column]].dropna().drop_duplicates()
    for isin, identifier in links.itertuples(index=False, name=None):
        first_isin = first_isin_by_id.setdefault(identifier, isin)
        parent[company_root(isin)] = company_root(first_isin)

company_by_isin = {isin: company_root(isin) for isin in parent}
company = window["ISIN"].map(company_by_isin)
prices = pd.to_numeric(window["AdjustedPrice"].str.replace(",", ".", regex=False), errors="coerce")
price_values = prices.to_numpy(dtype=float, na_value=np.nan)
has_price = np.isfinite(price_values) & (price_values > 0)
company_dates = pd.DataFrame({
    "Company": company[has_price],
    "Session": trading_dates.get_indexer(window.loc[has_price, "Date"]),
}).drop_duplicates().sort_values("Session")

n_dates = len(trading_dates)
eligible_companies = set()
for company_id, positions in company_dates.groupby("Company")["Session"]:
    # The endpoints include gaps before the first and after the last observation.
    missing_runs = np.diff(np.r_[-1, positions.to_numpy(), n_dates]) - 1
    if missing_runs.max() <= 5:
        eligible_companies.add(company_id)

eligible_rows = window.loc[company.isin(eligible_companies)]
print(f"Eligible: {len(eligible_companies):,} companies, "
      f"{eligible_rows['ISIN'].nunique():,} ISINs, {len(eligible_rows):,} rows.")

## Export eligible rows

Copy retained records directly to preserve source values, formatting, order and overlapping share classes.

In [ ]:
eligible_path.parent.mkdir(exist_ok=True)
keep_row = rows.index.isin(eligible_rows.index)

with NamedTemporaryFile(dir=eligible_path.parent, suffix=".tmp", delete=False) as dst:
    temporary = Path(dst.name)
    try:
        with raw_source.open("rb") as src:
            dst.write(src.readline())
            for line, keep in zip(src, keep_row, strict=True):
                if keep:
                    dst.write(line)
        dst.close()
        temporary.replace(eligible_path)
    finally:
        temporary.unlink(missing_ok=True)

print(f"Wrote {eligible_path}")

## Map eligible stocks to ICB sectors

Add Industry, Supersector, Sector and Subsector from `Data/stock_sector_mapping.csv`. The saved ICB snapshot is applied to the full price history.

Match by ISIN first. For an ISIN without a classification, use a shared `CompanyId` or `Internal code` when the mapped ISINs agree on all four labels.

In [ ]:
mapped_path = eligible_dir / "data_mapped.csv"
levels = [f"icb_{level}_name" for level in ("industry", "supersector", "sector", "subsector")]

identifiers = ["CompanyId", "Internal code"]
eligible = pd.read_csv(eligible_path, sep=";", usecols=["ISIN", *identifiers], dtype="string")
snapshot = pd.read_csv(data_dir / "stock_sector_mapping.csv", usecols=["ISIN", *levels], dtype="string")
by_isin = snapshot.set_index("ISIN", verify_integrity=True).dropna(subset=levels)

keys = ["Identifier type", "Identifier"]
isin_identifiers = eligible.melt(
    id_vars="ISIN", value_vars=identifiers, var_name=keys[0], value_name=keys[1]
).dropna().drop_duplicates()
mapped_identifiers = isin_identifiers.join(by_isin, on="ISIN").dropna(subset=levels)
labels_by_identifier = mapped_identifiers[[*keys, *levels]].drop_duplicates()
inherited = isin_identifiers.merge(labels_by_identifier, on=keys)[["ISIN", *levels]].drop_duplicates()
inherited = inherited.drop_duplicates("ISIN", keep=False).set_index("ISIN")

eligible_isins = pd.Index(eligible["ISIN"].unique(), name="ISIN")
exact_matches = by_isin.reindex(eligible_isins)
classifications = exact_matches.combine_first(inherited.reindex(eligible_isins))[levels]
exact_count = exact_matches[levels[0]].count()
fallback_count = classifications[levels[0]].count() - exact_count
unmapped = classifications.index[classifications[levels[0]].isna()]

print(f"ISINs: {exact_count} exact matches, {fallback_count} linked matches, {len(unmapped)} unmapped.")
if len(unmapped):
    print("Unmapped ISINs:", ", ".join(unmapped))

### Export `data_mapped.csv`

Append the four ICB labels to each source row. Keep the original `Sector` column and leave unresolved labels blank.

In [ ]:
labels_by_isin = {
    isin: labels for isin, *labels in classifications.fillna("").itertuples(name=None)
}

with NamedTemporaryFile(
    mode="w", encoding="utf-8", newline="", dir=eligible_dir, suffix=".tmp", delete=False
) as dst:
    temporary = Path(dst.name)
    try:
        with eligible_path.open(encoding="utf-8", newline="") as src:
            reader = csv.reader(src, delimiter=";")
            writer = csv.writer(dst, delimiter=";", lineterminator="\n")
            header = next(reader)
            isin_index = header.index("ISIN")
            writer.writerow([*header, *levels])
            for row in reader:
                writer.writerow([*row, *labels_by_isin[row[isin_index]]])
        dst.close()
        temporary.replace(mapped_path)
    finally:
        temporary.unlink(missing_ok=True)

print(f"Wrote {len(eligible):,} rows to {mapped_path}.")

## Reconstruct adjusted-price histories

Read `Data/eligible/data_mapped.csv`. Join nonoverlapping ISINs using known continuity links or stable `CompanyId` and `SecurityId` pairs. Also join successive ISINs with a shared `Internal code` when they meet on adjacent trading sessions and agree on the mapped ICB labels.

In [ ]:
histories_path = eligible_dir / "price_histories.csv"
icb_columns = [f"icb_{level}_name" for level in ("industry", "supersector", "sector", "subsector")]
columns = ["Date", "ISIN", "Name", "Sector", "CompanyId", "SecurityId", "Internal code",
           "AdjustedPrice", "Turnover", "Currency", *icb_columns]
data = pd.read_csv(mapped_path, sep=";", decimal=",", usecols=columns, parse_dates=["Date"])
data = data.sort_values(["Date", "ISIN", "Sector"], kind="stable")
assert data[["Date", "ISIN"]].notna().all().all()
assert data["Currency"].dropna().eq("NOK").all()
sessions = pd.DatetimeIndex(data["Date"].unique(), name="Date")
session_number = pd.Series(np.arange(len(sessions)), index=sessions)


def distinct_ids(values: pd.Series) -> tuple:
    return tuple(sorted(values.dropna().unique()))


profiles = data.groupby("ISIN").agg(
    first=("Date", "min"), last=("Date", "max"), company=("CompanyId", distinct_ids),
    security=("SecurityId", distinct_ids), internal=("Internal code", distinct_ids),
)
labels = data.groupby("ISIN")[icb_columns].last()
histories = {isin: {isin} for isin in profiles.index}


def join_histories(isins: list[str], *, adjacent: bool = False) -> bool:
    """Join compatible ISIN spans while keeping simultaneous classes separate."""
    members = set().union(*(histories[isin] for isin in isins))
    spans = profiles.loc[sorted(members)].sort_values("first")
    starts = spans["first"].iloc[1:].map(session_number).to_numpy()
    ends = spans["last"].iloc[:-1].map(session_number).to_numpy()
    if not (starts > ends).all():
        return False
    if adjacent:
        groups = [id(histories[isin]) for isin in spans.index]
        cross_component = np.array([a != b for a, b in zip(groups[:-1], groups[1:])])
        span_labels = labels.loc[spans.index]
        if (
            not (starts[cross_component] == ends[cross_component] + 1).all()
            or span_labels.isna().any().any()
            or len(span_labels.drop_duplicates()) != 1
        ):
            return False
    for isin in members:
        histories[isin] = members
    return True


known_links = [
    ["NO0003043309", "NO0013536151"],  # Kongsberg
    ["BMG368191034", "BMG3682E1012", "BMG3682E1921", "CY0200352116"],  # Frontline
    ["NO0005668905", "NO0012470089"],  # Tomra
]
for aliases in known_links:
    found = [isin for isin in aliases if isin in profiles.index]
    if len(found) > 1:
        assert join_histories(found), found

for keys, adjacent in [(["company", "security"], False), (["internal"], True)]:
    stable = profiles.loc[profiles[keys].map(len).eq(1).all(axis=1)]
    for _, group in stable.groupby(keys):
        if len(group) > 1:
            join_histories(group.index.tolist(), adjacent=adjacent)

stock_by_isin = {}
for isin, members in histories.items():
    spans = profiles.loc[sorted(members)].sort_values("last")
    assert spans["first"].iloc[1:].gt(spans["last"].shift().iloc[1:]).all(), isin
    stock_by_isin[isin] = spans.index[-1]
data["stock_id"] = data["ISIN"].map(stock_by_isin)
assert data.groupby(["stock_id", "Date"])["ISIN"].nunique().eq(1).all()
print(f"Linked {len(stock_by_isin)} ISINs into {data['stock_id'].nunique()} price histories.")

### Keep one share class per company

Group histories transitively by shared nonmissing CompanyId or Internal code. Keep the class with the most usable adjusted-price dates, then the highest summed NOK turnover, then the lowest stock ID. Export one row per retained history and trading session with gaps left blank.

In [ ]:
keys = ["stock_id", "Date"]
grouped = data.groupby(keys)
records = data.drop_duplicates(keys).set_index(keys).sort_index()
records["duplicate_rows"] = grouped.size() - 1
records["price_conflict"] = grouped["AdjustedPrice"].nunique(dropna=False).gt(1)
price = records["AdjustedPrice"]
records["invalid_price"] = ~(np.isfinite(price) & price.gt(0))
records["AdjustedPrice"] = price.where(~(records["price_conflict"] | records["invalid_price"]))
turnover = records["Turnover"]
records["Turnover"] = turnover.where(
    grouped["Turnover"].nunique(dropna=False).eq(1) & np.isfinite(turnover) & turnover.ge(0)
)

# Use issuer identifiers for class selection without combining class prices.
issuer_members = {stock_id: {stock_id} for stock_id in records.index.unique("stock_id")}
for identifier in ("CompanyId", "Internal code"):
    links = data[["stock_id", identifier]].dropna().drop_duplicates()
    for _, stock_group in links.groupby(identifier)["stock_id"]:
        members = set().union(*(issuer_members[stock_id] for stock_id in stock_group))
        for stock_id in members:
            issuer_members[stock_id] = members

class_ranking = records.groupby(level="stock_id").agg(
    usable_dates=("AdjustedPrice", "count"), total_turnover=("Turnover", "sum")
).reset_index()
class_ranking["issuer"] = class_ranking["stock_id"].map(
    {stock_id: min(members) for stock_id, members in issuer_members.items()}
)
class_ranking = class_ranking.sort_values(
    ["usable_dates", "total_turnover", "stock_id"], ascending=[False, False, True]
)
retained = class_ranking.drop_duplicates("issuer")
keep = retained["stock_id"]
contested = class_ranking.loc[class_ranking["issuer"].duplicated(keep=False)].copy()
contested["retained"] = contested["stock_id"].isin(keep)
display(contested)

records = records.loc[records.index.get_level_values("stock_id").isin(keep)]
by_stock = data.loc[data["stock_id"].isin(keep)].groupby("stock_id")
metadata = records.groupby(level="stock_id")[["Name", "CompanyId", "Sector", *icb_columns]].last()
metadata["sector_changed"] = by_stock["Sector"].nunique().gt(1)
grid = pd.MultiIndex.from_product([metadata.index, sessions], names=keys)
fields = ["ISIN", "AdjustedPrice", "Turnover", "duplicate_rows", "price_conflict", "invalid_price"]
panel = records[fields].reindex(grid).join(metadata).reset_index()
panel["duplicate_rows"] = panel["duplicate_rows"].fillna(0).astype(int)
for field in ("price_conflict", "invalid_price"):
    panel[field] = panel[field].eq(True)

previous = records.groupby(level="stock_id")[["ISIN", "AdjustedPrice"]].shift()
changed = previous["ISIN"].notna() & previous["ISIN"].ne(records["ISIN"])
transitions = records.loc[changed, ["ISIN", "AdjustedPrice"]].join(previous.add_prefix("previous_"))
transitions["price_change"] = transitions["AdjustedPrice"] / transitions["previous_AdjustedPrice"] - 1
print(f"{len(transitions)} ISIN transitions; {transitions['price_change'].abs().ge(.20).sum()} changes of 20% or more.")
display(transitions)

assert not panel.duplicated(keys).any()
assert panel["ISIN"].notna().sum() == len(records)
with NamedTemporaryFile(dir=histories_path.parent, suffix=".tmp", delete=False) as dst:
    temporary = Path(dst.name)
try:
    panel.to_csv(temporary, index=False)
    temporary.replace(histories_path)
finally:
    temporary.unlink(missing_ok=True)
print(f"Wrote {len(panel):,} rows for {len(metadata)} histories across {len(sessions):,} sessions to {histories_path}.")